<!-- field-paper-role-banner -->
> 🟩 **정답·해설본** — 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 수식과 tensor shape를 설명하세요.

# 지식 증류 · 모델 경량화 · 03. TinyBERT

## TinyBERT: Distilling BERT for Natural Language Understanding

- **원 논문:** [TinyBERT: Distilling BERT for Natural Language Understanding](https://aclanthology.org/2020.findings-emnlp.372/)
- **저자 / 발표:** Xiaoqi Jiao et al. · Findings of EMNLP 2020 (2020)
- **난이도 / 예상 시간:** 중급 · 약 75분
- **선수 지식:** PyTorch tensor와 autograd, softmax·cross entropy, 선형대수 기초
- **로컬 학습 데이터:** `data/field_curriculum/compression_bench.npz`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** Transformer student가 teacher의 hidden state, attention matrix, prediction을 함께 모방하도록 학습한다.

**축소하거나 생략한 부분:** 원 논문은 BERT-Base를 4/6-layer TinyBERT로 2단계 증류한다. 실습은 저장된 6-token teacher tensor와 작은 projection이다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | Figure 1: general/task-specific distillation | Task 3 | 같은 sequence target으로 general/task-specific 2단계 loss |
| §3.1, Eq. (4)–(5): embedding/hidden loss | Task 1 | regressor MSE |
| §3.1, Eq. (6): attention loss | Task 2 | attention matrix MSE |
| §3.1, Eq. (7)–(8): prediction/total loss | Task 3 | loss 감소 |

## 미니 재현
TinyBERT는 logits 하나만 복사하지 않고 Transformer의 hidden과 attention까지 여러 층에서 맞춥니다.

## 포트폴리오 해설: 정확도와 효율의 계약

$$\mathcal L_{model}=\sum_m(\mathcal L_{attn}^m+\mathcal L_{hidn}^m)+\mathcal L_{embd}+
\mathcal L_{pred}$$

- **기호 정의:** m은 student layer, L_attn/L_hidn/L_pred는 attention·hidden·prediction 손실입니다.
- **수식의 역할:** Transformer 내부의 여러 수준을 동시에 맞춰 작은 BERT의 표현 경로를 보존합니다.
- **구현 이유:** 정확도만 남기면 경량화의 비용 절감 주장을 검증할 수 없으므로,
  핵심 변환과 품질·비용 측정을 독립 함수와 method로 분리했습니다.
- **Task/코드 대응:** Task P의 논문 전용 class와 `forward`, `compute_loss`,
  `training_step`, `fit_portfolio_model`, `evaluate_portfolio_model`이
  Figure 1: general/task-specific distillation의 흐름을 명시합니다.
- **입출력 shape:** token embedding [B, 6, 16] → hidden [B, 6, 24], attention [B, 6, 6]
- **평가:** hidden/attention/logit loss와 분류 accuracy
- **원문 대비 한계:** 원 논문은 BERT-Base를 4/6-layer TinyBERT로 2단계 증류한다. 실습은 저장된 6-token teacher
tensor와 작은 projection이다.

경량화 실험에서는 정확도만 보지 않고 parameter·sparsity·bit·latency 중 논문이
주장한 비용을 함께 기록해야 합니다. 아래 코드는 단계별 중간값을 남깁니다.

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** tokens [B,T,D] → hidden/attention [B,T,H]/[B,T,T] → logits [B,K]
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

In [ ]:
from pathlib import Path
import math
import time
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import Tensor, nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(20260814)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
torch.set_default_device(DEVICE)
print(ACCELERATOR.summary())


def locate(relative):
    for base in (Path.cwd(), *Path.cwd().parents):
        path = base / relative
        if path.exists():
            return path
    raise FileNotFoundError(relative)


dataset_path = locate("data/field_curriculum/compression_bench.npz")
raw = np.load(dataset_path, allow_pickle=False)
features = torch.from_numpy(raw["features"]).float().to(DEVICE)
labels = torch.from_numpy(raw["labels"]).long().to(DEVICE)
teacher_logits = torch.from_numpy(raw["teacher_logits"]).float().to(DEVICE)
teacher_weights = torch.from_numpy(raw["teacher_weights"]).float().to(DEVICE)
calibration = torch.from_numpy(raw["calibration_activations"]).float().to(DEVICE)
dense_weights = torch.from_numpy(raw["dense_weights"]).float().to(DEVICE)
train_idx = torch.from_numpy(raw["train_idx"]).long().to(DEVICE)
test_idx = torch.from_numpy(raw["test_idx"]).long().to(DEVICE)
images = torch.from_numpy(raw["images"]).float().to(DEVICE)
image_labels = torch.from_numpy(raw["image_labels"]).long().to(DEVICE)
image_train_idx = torch.from_numpy(raw["image_train_idx"]).long().to(DEVICE)
image_test_idx = torch.from_numpy(raw["image_test_idx"]).long().to(DEVICE)
token_embeddings = torch.from_numpy(raw["token_embeddings"]).float().to(DEVICE)
teacher_hidden = torch.from_numpy(raw["teacher_hidden"]).float().to(DEVICE)
teacher_attention = torch.from_numpy(raw["teacher_attention"]).float().to(DEVICE)
candidate_configs = torch.from_numpy(raw["candidate_configs"]).float()
assert features.shape == (512, 32) and teacher_logits.shape == (512, 4)
assert images.shape == (320, 1, 16, 16)
assert teacher_hidden.shape == (256, 6, 24)
print(
    dataset_path.name, "features", tuple(features.shape), "images", tuple(images.shape)
)

### 구현 단계 2 · 목적함수·학습 update

- **원문 위치:** §3.1, Eq. (4)–(5): embedding/hidden loss
- **이 셀의 책임:** Task 1
- **Tensor shape 계약:** tokens [B,T,D] → hidden/attention [B,T,H]/[B,T,T] → logits [B,K]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** regressor MSE. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
class TinyStudent(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Linear(16, 12)
        self.regressor = nn.Linear(12, 24)
        self.q = nn.Linear(12, 12)
        self.k = nn.Linear(12, 12)
        self.head = nn.Linear(12, 4)

    def forward(self, x):
        hidden = torch.tanh(self.embed(x))
        q, k = self.q(hidden), self.k(hidden)
        attention = F.softmax(q @ k.transpose(1, 2) / math.sqrt(12), -1)
        return hidden, self.regressor(hidden), attention, self.head(hidden.mean(1))


student = TinyStudent()
hidden, regressed, _, _ = student(token_embeddings[:4])
hidden_loss = F.mse_loss(regressed, teacher_hidden[:4])
assert hidden.shape == (4, 6, 12) and regressed.shape == (4, 6, 24) and hidden_loss > 0

### 구현 단계 3 · 핵심 연산·모델

- **원문 위치:** §3.1, Eq. (6): attention loss
- **이 셀의 책임:** Task 2
- **Tensor shape 계약:** tokens [B,T,D] → hidden/attention [B,T,H]/[B,T,T] → logits [B,K]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** attention matrix MSE. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
_, _, student_attention, _ = student(token_embeddings[:8])
attention_loss = F.mse_loss(student_attention, teacher_attention[:8])
assert student_attention.shape == (8, 6, 6)
assert torch.allclose(student_attention.sum(-1), torch.ones(8, 6), atol=1e-6)
assert attention_loss > 0

### 구현 단계 4 · 평가·완료 증거

- **원문 위치:** Figure 1: general/task-specific distillation / §3.1, Eq. (4)–(5): embedding/hidden loss / §3.1, Eq. (6): attention loss / §3.1, Eq. (7)–(8): prediction/total loss
- **이 셀의 책임:** Task 3 / Task 1 / Task 2
- **Tensor shape 계약:** tokens [B,T,D] → hidden/attention [B,T,H]/[B,T,T] → logits [B,K]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** 같은 sequence target으로 general/task-specific 2단계 loss / regressor MSE / attention matrix MSE / loss 감소. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
sequence_labels = token_embeddings.mean(1)[:, :4].argmax(-1)
teacher_prediction = F.one_hot(sequence_labels, 4).float() * 4
optimizer = torch.optim.Adam(student.parameters(), lr=0.02)
losses = []
for _ in range(55):
    optimizer.zero_grad()
    _, regressed, attention, prediction = student(token_embeddings)
    loss = (
        F.mse_loss(regressed, teacher_hidden)
        + 8 * F.mse_loss(attention, teacher_attention)
        + F.kl_div(
            F.log_softmax(prediction / 2, -1),
            F.softmax(teacher_prediction / 2, -1),
            reduction="batchmean",
        )
        * 4
    )
    loss.backward()
    optimizer.step()
    losses.append(float(loss.detach()))
with torch.no_grad():
    accuracy = (
        (student(token_embeddings)[3].argmax(-1) == sequence_labels)
        .float()
        .mean()
        .item()
    )
assert losses[-1] < losses[0] * 0.55 and accuracy > 0.65
plt.plot(losses)
plt.title(f"TinyBERT tensor distillation, acc={accuracy:.1%}")
plt.show()
print(f"loss {losses[0]:.3f}→{losses[-1]:.3f}")

### Task P1 · 압축 대상과 핵심 모델

tensor shape와 비용 단위를 먼저 확인한 뒤 논문 전용 class의 `forward`와
`compute_loss`에 정확도·표현 보존 계약을 명시합니다.

### 구현 단계 5 · 핵심 연산·모델

- **원문 위치:** Figure 1: general/task-specific distillation / §3.1, Eq. (4)–(5): embedding/hidden loss / §3.1, Eq. (6): attention loss / §3.1, Eq. (7)–(8): prediction/total loss
- **이 셀의 책임:** Task 3 / Task 1 / Task 2
- **Tensor shape 계약:** tokens [B,T,D] → hidden/attention [B,T,H]/[B,T,T] → logits [B,K]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 같은 sequence target으로 general/task-specific 2단계 loss / regressor MSE / attention matrix MSE / loss 감소. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class PortfolioConfig:
    hidden_dim: int = 12
    temperature: float = 2.0
    learning_rate: float = 0.02
    general_steps: int = 8
    task_steps: int = 8


def sequence_teacher_targets(inputs: Tensor) -> tuple[Tensor, Tensor]:
    """Derive task labels and dark-knowledge logits from each input sequence."""
    class_count = teacher_logits.shape[-1]
    pooled = inputs.mean(dim=1)
    predictions = 4.0 * pooled[:, :class_count]
    return predictions.argmax(dim=1), predictions


def prepare_batch() -> tuple[Tensor, Tensor, Tensor, Tensor, Tensor]:
    inputs = token_embeddings[:192]
    targets, prediction_targets = sequence_teacher_targets(inputs)
    return (
        inputs,
        targets,
        prediction_targets,
        teacher_hidden[:192],
        teacher_attention[:192],
    )


class TinybertPortfolioModel(nn.Module):
    def __init__(self, config: PortfolioConfig):
        super().__init__()
        self.config = config
        input_dim = token_embeddings.shape[-1]
        self.query = nn.Linear(input_dim, config.hidden_dim)
        self.key = nn.Linear(input_dim, config.hidden_dim)
        self.value = nn.Linear(input_dim, config.hidden_dim)
        self.hidden_regressor = nn.Linear(config.hidden_dim, teacher_hidden.shape[-1])
        self.classifier = nn.Linear(config.hidden_dim, teacher_logits.shape[-1])

    def forward(self, inputs: Tensor) -> tuple[Tensor, Tensor, Tensor]:
        query = self.query(inputs)
        key = self.key(inputs)
        value = self.value(inputs)
        scale = self.config.hidden_dim**0.5
        attention = torch.softmax(query @ key.transpose(1, 2) / scale, dim=-1)
        hidden = attention @ value
        regressed = self.hidden_regressor(hidden)
        logits = self.classifier(hidden.mean(dim=1))
        return logits, regressed, attention

    def compute_loss(
        self,
        outputs: tuple[Tensor, Tensor, Tensor],
        targets: Tensor,
        prediction_target: Tensor,
        hidden_target: Tensor,
        attention_target: Tensor,
        phase: str,
    ) -> Tensor:
        logits, hidden, attention = outputs
        hidden_loss = F.mse_loss(hidden, hidden_target)
        attention_loss = F.mse_loss(attention, attention_target)
        intermediate = hidden_loss + 4.0 * attention_loss
        if phase == "general":
            return intermediate
        if phase != "task_specific":
            raise ValueError(f"unknown TinyBERT phase: {phase}")
        temperature = self.config.temperature
        teacher_probability = torch.softmax(prediction_target / temperature, dim=1)
        soft_prediction = F.kl_div(
            F.log_softmax(logits / temperature, dim=1),
            teacher_probability,
            reduction="batchmean",
        )
        prediction = F.cross_entropy(logits, targets) + temperature**2 * soft_prediction
        return prediction + intermediate

    def training_step(
        self,
        inputs: Tensor,
        targets: Tensor,
        prediction_target: Tensor,
        hidden_target: Tensor,
        attention_target: Tensor,
        phase: str,
    ) -> Tensor:
        outputs = self(inputs)
        return self.compute_loss(
            outputs,
            targets,
            prediction_target,
            hidden_target,
            attention_target,
            phase,
        )

### Task P2 · 학습·pruning·distillation update

`fit_portfolio_model`에서 optimizer, mask/EMA/quantization 경계와 update 순서를
드러냅니다. 비용 절감이 학습 경로에 미치는 영향을 함께 읽으세요.

### 구현 단계 6 · 목적함수·학습 update

- **원문 위치:** Figure 1: general/task-specific distillation / §3.1, Eq. (4)–(5): embedding/hidden loss / §3.1, Eq. (6): attention loss / §3.1, Eq. (7)–(8): prediction/total loss
- **이 셀의 책임:** Task 3 / Task 1 / Task 2
- **Tensor shape 계약:** tokens [B,T,D] → hidden/attention [B,T,H]/[B,T,T] → logits [B,K]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** 같은 sequence target으로 general/task-specific 2단계 loss / regressor MSE / attention matrix MSE / loss 감소. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def fit_portfolio_model(model: TinybertPortfolioModel) -> list[float]:
    batch = prepare_batch()
    history = []
    intermediate_parameters = [
        *model.query.parameters(),
        *model.key.parameters(),
        *model.value.parameters(),
        *model.hidden_regressor.parameters(),
    ]
    general_optimizer = torch.optim.Adam(
        intermediate_parameters,
        lr=model.config.learning_rate,
    )
    for _ in range(model.config.general_steps):
        loss = model.training_step(*batch, phase="general")
        general_optimizer.zero_grad()
        loss.backward()
        general_optimizer.step()
        history.append(float(loss.detach().cpu()))
    task_optimizer = torch.optim.Adam(
        model.parameters(),
        lr=model.config.learning_rate,
    )
    for _ in range(model.config.task_steps):
        loss = model.training_step(*batch, phase="task_specific")
        task_optimizer.zero_grad()
        loss.backward()
        task_optimizer.step()
        history.append(float(loss.detach().cpu()))
    return history

### Task P3 · 품질과 비용을 함께 평가

accuracy 또는 reconstruction 품질과 parameter·sparsity·bit·latency를 별도로
기록합니다. 로컬 지표는 원문 하드웨어 benchmark의 대체값이 아닙니다.

### 구현 단계 7 · 평가·완료 증거

- **원문 위치:** §3.1, Eq. (7)–(8): prediction/total loss
- **이 셀의 책임:** Task 3
- **Tensor shape 계약:** tokens [B,T,D] → hidden/attention [B,T,H]/[B,T,T] → logits [B,K]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** loss 감소. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def evaluate_portfolio_model(model: TinybertPortfolioModel) -> dict[str, float]:
    with torch.no_grad():
        test_inputs = token_embeddings[192:256]
        test_targets, prediction_targets = sequence_teacher_targets(test_inputs)
        logits, hidden, attention = model(test_inputs)
        accuracy = (logits.argmax(1) == test_targets).float().mean()
        hidden_mse = F.mse_loss(hidden, teacher_hidden[192:256])
        attention_mse = F.mse_loss(attention, teacher_attention[192:256])
        temperature = model.config.temperature
        prediction_kl = F.kl_div(
            F.log_softmax(logits / temperature, dim=1),
            torch.softmax(prediction_targets / temperature, dim=1),
            reduction="batchmean",
        )
    return {
        "test_accuracy": float(accuracy.cpu()),
        "prediction_kl": float(prediction_kl.cpu()),
        "hidden_mse": float(hidden_mse.cpu()),
        "attention_mse": float(attention_mse.cpu()),
        "two_stage_updates": float(
            model.config.general_steps + model.config.task_steps
        ),
    }


portfolio_model = TinybertPortfolioModel(PortfolioConfig())
portfolio_history = fit_portfolio_model(portfolio_model)
portfolio_metrics = evaluate_portfolio_model(portfolio_model)
assert np.isfinite(portfolio_history).all()
assert portfolio_metrics["test_accuracy"] > 0.35
print(portfolio_metrics)

## 정답 해석 가이드

핵심은 최종 숫자를 외우는 것이 아니라 **원문 위치 → 수식 → 코드 → 반증 가능한 증거**를
연결하는 것입니다. 이 노트북은 다음 원 규모 조건을 재현하지 않았습니다:

> 원 논문은 BERT-Base를 4/6-layer TinyBERT로 2단계 증류한다. 실습은 저장된 6-token teacher tensor와 작은 projection이다.

다음 실험에서는 데이터 크기, 모델 폭, 학습 step 중 하나만 바꾸고 동일 seed에서 비교하세요.